In [0]:
bronze_count = spark.table("bronze_ais_canonical").count()
silver_count = spark.table("silver_ais_events").count()
gold_count = spark.table("gold_traffic_daily").count()

print(f"Bronze rows: {bronze_count}")
print(f"Silver rows: {silver_count}")
print(f"Gold rows: {gold_count}")

In [0]:
spark.sql("""
SELECT
    MIN(event_date) AS first_day,
    MAX(event_date) AS last_day,
    COUNT(*) AS total_days
FROM gold_traffic_daily
""").show()

In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_ais_canonical")
silver_df = spark.table("silver_ais_events")
reject_df = spark.table("bronze_ais_rejects")

bronze_count = bronze_df.count()

valid_raw_count = bronze_df.filter(
    (F.col("latitude").between(-90, 90)) &
    (F.col("longitude").between(-180, 180)) &
    (F.col("vessel_id").isNotNull()) &
    (F.col("event_timestamp").isNotNull())
).count()

reject_count = reject_df.count()
silver_count = silver_df.count()

duplicate_count = valid_raw_count - silver_count

print(f"Bronze count: {bronze_count}")
print(f"Valid raw count: {valid_raw_count}")
print(f"Reject count: {reject_count}")
print(f"Silver count (after dedup): {silver_count}")
print(f"Duplicates removed in Silver: {duplicate_count}")
print(f"Valid raw + Reject: {valid_raw_count + reject_count}")

In [0]:
display(
    spark.table("gold_traffic_daily")
    .orderBy("event_date")
)

In [0]:
from pyspark.sql import functions as F

bronze_count = spark.table("bronze_ais_canonical").count()
silver_count = spark.table("silver_ais_events").count()
reject_count = spark.table("bronze_ais_rejects").count()
gold_count = spark.table("gold_traffic_daily").count()

reject_percentage = round(
    (reject_count / bronze_count) * 100,
    2
)

quality_summary = spark.createDataFrame(
    [
        (
            bronze_count,
            silver_count,
            reject_count,
            gold_count,
            reject_percentage
        )
    ],
    [
        "bronze_count",
        "silver_count",
        "reject_count",
        "gold_count",
        "reject_percentage"
    ]
)
# quality_summary = quality_summary.withColumn
# (
#     "reject_percentage_str",
#     F.concat
#     (
#         F.format_number(F.col("reject_percentage"), 2),
#         F.lit("%")
#     )
# )

display(quality_summary)

In [0]:
quality_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("quality_summary")